# Student Placement Prediction - ML Notebook
**Goal:** predict whether a student gets placed (classification) and the expected salary of placed students (regression).

**Steps:** 1) Load data 2) EDA 3) Preprocess + train/test split 4) Train models 5) Tune with GridSearchCV 6) Evaluate (accuracy, precision, recall, F1, confusion matrix, ROC) 7) Feature importance 8) Salary regression 9) Export for the website

**Dataset:** `placement_data.csv` - 1500 **synthetic** students generated by `data/generate_dataset.py` with the same 10 fields the website collects (CGPA, 10th, 12th, aptitude, communication, programming, projects, internships, certifications, backlogs). It is NOT real data: results show how well models learn the generating rule. To use real records, replace the CSV (columns above + `placed` 1/0 + optional `salary`) and re-run.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)

CSV_PATH = 'placement_data.csv'   # change if your file has another name
SEED = 42

## 1. Load the data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isna().sum())
df.describe()

## 2. Exploratory Data Analysis (EDA)
We check the class balance, how each feature differs between placed / not placed students, and how features correlate.

In [ ]:
FEATURES = ['cgpa','tenth','twelfth','aptitude','communication','programming','projects','internships','certifications','backlogs']
print(df['placed'].value_counts())
df['placed'].map({1:'Placed',0:'Not placed'}).value_counts().plot(kind='bar', color=['0.3','0.65'], title='Placed vs Not placed')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for ax, c in zip(axes.ravel(), FEATURES):
    for label, colour in [(1,'0.2'), (0,'0.7')]:
        ax.hist(df.loc[df['placed']==label, c], bins=12, alpha=.6, color=colour, label='Placed' if label else 'Not placed')
    ax.set_title(c)
axes[0,0].legend(); plt.tight_layout(); plt.show()

In [ ]:
corr = df[FEATURES + ['placed']].corr()
plt.figure(figsize=(6,5)); plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=60); plt.yticks(range(len(corr)), corr.columns)
plt.title('Correlation matrix'); plt.show()
corr['placed'].sort_values(ascending=False)

## 3. Preprocessing and train-test split
All 10 features are numeric, so no encoding is needed (scaling happens inside the Logistic Regression pipeline only).
We use a **stratified 80/20 split** so both parts keep the same placed/not-placed ratio. The test set is kept untouched until the final evaluation.

In [ ]:
df = df.dropna(subset=FEATURES + ['placed']).reset_index(drop=True)
X = df[FEATURES].copy()
y = df['placed'].astype(int)

idx_tr, idx_te = train_test_split(np.arange(len(X)), test_size=0.2, stratify=y, random_state=SEED)
X_tr, X_te, y_tr, y_te = X.iloc[idx_tr], X.iloc[idx_te], y.iloc[idx_tr], y.iloc[idx_te]
print('train:', X_tr.shape, ' test:', X_te.shape)

## 4. Models
* **Logistic Regression** - linear model that outputs a probability; needs scaled features.
* **Decision Tree** - if-else rules learned from data; easy to explain but can overfit.
* **Random Forest** - many trees voting together; usually more stable than one tree.
* **Gradient Boosting** - trees built one after another, each fixing the previous one's mistakes.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
baselines = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(max_iter=2000, random_state=SEED))]),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED),
}
for name, m in baselines.items():
    print(f"{name:20s} 5-fold CV F1 on train data: {cross_val_score(m, X_tr, y_tr, cv=cv, scoring='f1').mean():.3f}")

## 5. Hyperparameter tuning (GridSearchCV)
Grid search tries every parameter combination with 5-fold cross-validation **on the training data only**, so the test set never influences the choice.

In [ ]:
grids = {
    'Logistic Regression': (baselines['Logistic Regression'], {'m__C': [0.1, 1, 10]}),
    'Decision Tree': (DecisionTreeClassifier(random_state=SEED), {'max_depth':[3,4,5,6], 'min_samples_leaf':[1,3,5]}),
    'Random Forest': (RandomForestClassifier(random_state=SEED), {'n_estimators':[100,200], 'max_depth':[4,6,8], 'min_samples_leaf':[1,3]}),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=SEED), {'n_estimators':[100,200], 'max_depth':[2,3], 'learning_rate':[0.05,0.1]}),
}
tuned, rows = {}, []
for name, (est, grid) in grids.items():
    gs = GridSearchCV(est, grid, cv=cv, scoring='f1').fit(X_tr, y_tr)
    tuned[name] = gs.best_estimator_
    pred = gs.best_estimator_.predict(X_te); proba = gs.best_estimator_.predict_proba(X_te)[:,1]
    rows.append({'Model':name, 'Best params':gs.best_params_, 'CV F1':round(gs.best_score_,3),
                 'Accuracy':round(accuracy_score(y_te,pred),3), 'Precision':round(precision_score(y_te,pred),3),
                 'Recall':round(recall_score(y_te,pred),3), 'F1':round(f1_score(y_te,pred),3),
                 'ROC-AUC':round(roc_auc_score(y_te,proba),3)})
results = pd.DataFrame(rows)
results

## 6. Evaluation on the test set
* **Accuracy** - fraction of correct predictions.
* **Precision** - of students predicted "placed", how many really were.
* **Recall** - of students who really were placed, how many we found.
* **F1** - balance of precision and recall.
* **Confusion matrix** - counts of TP, FP, TN, FN.
The best model is chosen using **cross-validation F1** (not the test score).

In [ ]:
best_name = results.sort_values('CV F1', ascending=False).iloc[0]['Model']
best = tuned[best_name]; best_pred = best.predict(X_te)
print('Best model:', best_name)
print(classification_report(y_te, best_pred, target_names=['Not placed','Placed']))
cm = confusion_matrix(y_te, best_pred)
plt.imshow(cm, cmap='Greys')
for (i,j),v in np.ndenumerate(cm): plt.text(j,i,v,ha='center',va='center',color='white' if v>cm.max()/2 else 'black',fontsize=14)
plt.xticks([0,1],['Not placed','Placed']); plt.yticks([0,1],['Not placed','Placed'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix - '+best_name); plt.show()

In [ ]:
for name, m in tuned.items():
    fpr, tpr, _ = roc_curve(y_te, m.predict_proba(X_te)[:,1])
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_te, m.predict_proba(X_te)[:,1]):.2f})")
plt.plot([0,1],[0,1],'k--'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.legend(); plt.title('ROC curves'); plt.show()

## 7. Feature importance
Permutation importance shuffles one feature at a time and measures how much the F1-score drops - a bigger drop means the model relies on that feature more.

In [ ]:
pi = permutation_importance(best, X_te, y_te, n_repeats=30, random_state=SEED, scoring='f1')
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values()
imp.plot(kind='barh', color='0.4', title='Permutation importance'); plt.show()
imp.sort_values(ascending=False)

## 8. Salary prediction (regression)
For placed students only, predict the salary with Linear Regression and Random Forest Regressor.
Metrics: **MAE** (average error), **RMSE** (penalises big errors), **R2** (1 = perfect, 0 = no better than the mean).

In [ ]:
if 'salary' in df.columns:
    sal = pd.to_numeric(df['salary'], errors='coerce')
    placed = np.where((y==1) & sal.notna())[0]
    tr = np.array([i for i in placed if i in set(idx_tr)]); te = np.array([i for i in placed if i in set(idx_te)])
    regs = {'Linear Regression': Pipeline([('sc',StandardScaler()),('m',LinearRegression())]),
            'Random Forest Regressor': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=3, random_state=SEED)}
    out = []
    for name, r in regs.items():
        cv_r2 = cross_val_score(r, X.iloc[tr], sal.iloc[tr], cv=5, scoring='r2').mean()
        r.fit(X.iloc[tr], sal.iloc[tr]); p = r.predict(X.iloc[te])
        out.append({'Model':name, 'CV R2':round(cv_r2,3), 'MAE':round(mean_absolute_error(sal.iloc[te],p),2),
                    'RMSE':round(float(np.sqrt(mean_squared_error(sal.iloc[te],p))),2), 'R2':round(r2_score(sal.iloc[te],p),3)})
    display(pd.DataFrame(out)) if 'display' in globals() else print(pd.DataFrame(out))
else:
    print('No salary column in this dataset - skipping regression.')

## 9. Export for the website and the paper
This runs `train_model.py`, which repeats the same steps and saves the model, metrics, graphs and LaTeX tables in `output/`. The Flask backend loads `output/model.joblib` automatically.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'train_model.py', CSV_PATH], capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])